In [1]:
import pandas as pd
import numpy as np
from category_encoders import CatBoostEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, brier_score_loss

In [2]:
BALL_PATH = "C:/Desktop/crickanal/outputs/ball_by_ball_canonical.csv"
MATCH_PATH = "C:/Desktop/crickanal/outputs/match_master_clean_mergedwithclimate_teamnormalized.csv"

print("Loading data...")
df_ball = pd.read_csv(BALL_PATH, parse_dates=["date"])
df_match = pd.read_csv(MATCH_PATH, parse_dates=["date"])

# Merge winner
df_match_winner = df_match[["match_id", "winner"]].copy()
df_ball = df_ball.merge(df_match_winner, on="match_id", how="left")

Loading data...


joining winner from other csv winner

In [3]:
print("\nPreparing modeling dataset (both innings)...")

df_model = df_ball[
    (df_ball["has_elo"] == True) &
    (df_ball["winner"].notna())
].copy()

# Add is_chasing indicator
df_model["is_chasing"] = (df_model["innings"] == 2).astype(int)

# Fill 1st innings NaN values (no target/chase in innings 1)
df_model["required_run_rate"] = df_model["required_run_rate"].fillna(0)
df_model["req_per_ball"] = df_model["req_per_ball"].fillna(0)
df_model["runs_remaining"] = df_model["runs_remaining"].fillna(0)

# Fill 1st innings NaN values (no target/chase in innings 1)
df_model["required_run_rate"] = df_model["required_run_rate"].fillna(0)
df_model["req_per_ball"] = df_model["req_per_ball"].fillna(0)
df_model["runs_remaining"] = df_model["runs_remaining"].fillna(0)
df_model["rr_diff"] = df_model["rr_diff"].fillna(0)  # ADD THIS LINE


# Filter out invalid balls
df_model = df_model[
    (df_model["balls_remaining"] > 0) &
    (df_model["current_run_rate"].notna())
].copy()

# Create target: did batting team win?
df_model["target"] = (df_model["batting_team"] == df_model["winner"]).astype(int)

# Verify win rate
print(f"Overall win rate: {df_model['target'].mean():.3f}")

# Check last ball win rate (FIX 2: Sort before tail)
end_balls = (
    df_model
    .sort_values(["match_id", "innings", "over", "ball"])
    .groupby("match_id")
    .tail(1)
)
print(f"Last ball win rate: {end_balls['target'].mean():.3f}")


Preparing modeling dataset (both innings)...
Overall win rate: 0.491
Last ball win rate: 0.524


In [4]:
NUMERIC_FEATURES = [
    "runs_remaining",
    "balls_remaining",
    "wickets_lost",
    "current_run_rate",
    "required_run_rate",
    "req_per_ball",
    "rr_diff",
    "batting_elo",
    "bowling_elo",
    "elo_diff",
    "phase_powerplay",
    "phase_death",
    "is_chasing"
]

CATEGORICAL_FEATURES = [
    "venue",
    "league"
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [5]:
print("\nCreating chronological splits...")

match_dates = (
    df_model[["match_id", "date"]]
    .drop_duplicates()
    .sort_values("date")
)

train_cutoff = match_dates["date"].quantile(0.65)
val_cutoff = match_dates["date"].quantile(0.85)

train_df = df_model[df_model["date"] <= train_cutoff]
val_df = df_model[
    (df_model["date"] > train_cutoff) &
    (df_model["date"] <= val_cutoff)
]
test_df = df_model[df_model["date"] > val_cutoff]

def split_summary(df, name):
    print(f"\n{name}")
    print(f"  Matches: {df['match_id'].nunique()}")
    print(f"  Balls: {len(df)}")
    print(f"  Innings 1: {(df['innings']==1).sum()}")
    print(f"  Innings 2: {(df['innings']==2).sum()}")
    print(f"  Win rate: {df['target'].mean():.3f}")

split_summary(train_df, "TRAIN")
split_summary(val_df, "VAL")
split_summary(test_df, "TEST")


Creating chronological splits...

TRAIN
  Matches: 3799
  Balls: 882754
  Innings 1: 459499
  Innings 2: 423255
  Win rate: 0.490

VAL
  Matches: 1166
  Balls: 270535
  Innings 1: 141286
  Innings 2: 129249
  Win rate: 0.493

TEST
  Matches: 877
  Balls: 202894
  Innings 1: 106149
  Innings 2: 96745
  Win rate: 0.490


In [6]:
print("\nEncoding categorical features...")

encoder = CatBoostEncoder(cols=CATEGORICAL_FEATURES, random_state=42)

X_train = train_df[FEATURES]
y_train = train_df["target"]

X_val = val_df[FEATURES]
y_val = val_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

X_train_enc = encoder.fit_transform(X_train, y_train)
X_val_enc = encoder.transform(X_val)
X_test_enc = encoder.transform(X_test)

# Check for NaNs
nan_counts = X_train_enc.isna().sum()
if nan_counts.sum() > 0:
    print("\n⚠️ NaN counts after encoding:")
    print(nan_counts[nan_counts > 0])
else:
    print("✅ No NaNs after encoding")


Encoding categorical features...
✅ No NaNs after encoding


In [7]:
print("\nTraining win probability model...")

wp_model = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    n_jobs=-1,
    random_state=42
)

wp_model.fit(X_train_enc, y_train)


Training win probability model...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [8]:
print("Calibrating model...")

wp_model_cal = CalibratedClassifierCV(
    wp_model,
    method="isotonic",
    cv="prefit"
)

wp_model_cal.fit(X_val_enc, y_val)

# Evaluate
val_probs = wp_model_cal.predict_proba(X_val_enc)[:, 1]
test_probs = wp_model_cal.predict_proba(X_test_enc)[:, 1]

print("\n" + "="*60)
print("MODEL PERFORMANCE")
print("="*60)
print(f"VAL  LogLoss: {log_loss(y_val, val_probs):.4f}  |  Brier: {brier_score_loss(y_val, val_probs):.4f}")
print(f"TEST LogLoss: {log_loss(y_test, test_probs):.4f}  |  Brier: {brier_score_loss(y_test, test_probs):.4f}")

Calibrating model...

MODEL PERFORMANCE


C:\Users\kathe\AppData\Roaming\Python\Python314\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


VAL  LogLoss: 0.4428  |  Brier: 0.1442
TEST LogLoss: 0.4596  |  Brier: 0.1476


In [9]:
print("\nRetraining on train+val...")

X_trval_enc = pd.concat([X_train_enc, X_val_enc])
y_trval = pd.concat([y_train, y_val])

wp_model_final = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    n_jobs=-1,
    random_state=42
)

wp_model_final.fit(X_trval_enc, y_trval)

# Reuse existing calibrator (no recalibration)
wp_model_cal_final = wp_model_cal


Retraining on train+val...


In [11]:
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, accuracy_score

# If wp_model_cal_final was fit as CalibratedClassifierCV(base_estimator=wp_model, ...)
# using the same encoded features, just do:
y_test_proba = wp_model_cal_final.predict_proba(X_test_enc)[:, 1]

# Core probabilistic metrics
brier = brier_score_loss(y_test, y_test_proba)
ll    = log_loss(y_test, y_test_proba)
auc   = roc_auc_score(y_test, y_test_proba)

# Optional hard predictions at 0.5 threshold
y_test_pred = (y_test_proba >= 0.5).astype(int)
acc = accuracy_score(y_test, y_test_pred)

print("Win-probability (WP) model performance on test set:")
print(f"  Brier score : {brier:.4f}")
print(f"  Log loss    : {ll:.4f}")
print(f"  ROC-AUC     : {auc:.4f}")
print(f"  Accuracy    : {acc:.4f}")


Win-probability (WP) model performance on test set:
  Brier score : 0.1476
  Log loss    : 0.4596
  ROC-AUC     : 0.8679
  Accuracy    : 0.7883


In [12]:
from sklearn.metrics import brier_score_loss, log_loss

# Train, val, test probabilities from calibrated model
y_train_proba = wp_model_cal_final.predict_proba(X_train_enc)[:, 1]
y_val_proba   = wp_model_cal_final.predict_proba(X_val_enc)[:, 1]
y_test_proba  = wp_model_cal_final.predict_proba(X_test_enc)[:, 1]

# Train scores
train_logloss = log_loss(y_train, y_train_proba)
train_brier   = brier_score_loss(y_train, y_train_proba)

# Val scores
val_logloss = log_loss(y_val, y_val_proba)
val_brier   = brier_score_loss(y_val, y_val_proba)

# Test scores
test_logloss = log_loss(y_test, y_test_proba)
test_brier   = brier_score_loss(y_test, y_test_proba)

print(f"TRAIN LogLoss: {train_logloss:.4f}  |  Brier: {train_brier:.4f}")
print(f"VAL   LogLoss: {val_logloss:.4f}  |  Brier: {val_brier:.4f}")
print(f"TEST  LogLoss: {test_logloss:.4f}  |  Brier: {test_brier:.4f}")


TRAIN LogLoss: 0.4356  |  Brier: 0.1408
VAL   LogLoss: 0.4428  |  Brier: 0.1442
TEST  LogLoss: 0.4596  |  Brier: 0.1476


In [15]:
print("\nPredicting win probability for all balls...")

df_all = df_ball.copy()

# Include both innings
df_all = df_all[df_all["has_elo"] == True].copy()

# Add is_chasing indicator
df_all["is_chasing"] = (df_all["innings"] == 2).astype(int)

# Fill NaNs for 1st innings
df_all["required_run_rate"] = df_all["required_run_rate"].fillna(0)
df_all["req_per_ball"] = df_all["req_per_ball"].fillna(0)
df_all["runs_remaining"] = df_all["runs_remaining"].fillna(0)
df_all["rr_diff"] = df_all["rr_diff"].fillna(0)  # ADD THIS LINE
# Filter valid balls
df_all = df_all[
    (df_all["balls_remaining"] > 0) &
    (df_all["current_run_rate"].notna())
].copy()

# Encode and predict
X_all = df_all[FEATURES]
X_all_enc = encoder.transform(X_all)

df_all["win_probability"] = wp_model_cal_final.predict_proba(X_all_enc)[:, 1]

print(f"\n✅ Predicted WP for {len(df_all):,} balls")


Predicting win probability for all balls...

✅ Predicted WP for 1,356,183 balls


In [16]:
print("\nComputing WPA...")

df_all = df_all.sort_values(["match_id", "innings", "over", "ball"])

df_all["wpa"] = (
    df_all
    .groupby(["match_id", "innings"])["win_probability"]
    .diff()
    .fillna(0.0)
)


Computing WPA...


In [18]:
print("\n" + "="*60)
print("WPA VALIDATION")
print("="*60)

# FIX 1: Correct conservation comment
# Check WPA conservation (sum per match should ≈ 0.0 under innings-reset convention)
wpa_sum = df_all.groupby("match_id")["wpa"].sum()
print(f"\nWPA sum per match:")
print(f"  Mean: {wpa_sum.mean():.4f} (should be ≈ 0.0)")
print(f"  Std:  {wpa_sum.std():.4f}")
print(f"  Min:  {wpa_sum.min():.4f}")
print(f"  Max:  {wpa_sum.max():.4f}")

# Check innings distribution
print(f"\nBalls by innings:")
print(df_all.groupby("innings").size())

# Check WPA by innings
print(f"\nTotal WPA by innings:")
print(df_all.groupby("innings")["wpa"].sum())

# FIX 2: Terminal WP check with proper sorting
end_wp = (
    df_all
    .sort_values(["match_id", "innings", "over", "ball"])
    .groupby("match_id")
    .tail(1)["win_probability"]
)
print(f"\nTerminal WP (last ball of match):")
print(f"  Mean: {end_wp.mean():.3f} (should be ≈ 0.52)")
print(f"  Std:  {end_wp.std():.3f}")

# WPA distribution
print("\nWPA distribution (should have heavy tails):")
wpa_desc = df_all["wpa"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print(wpa_desc)

# Win probability gradient check
print("\nWin probability by balls remaining (first/last):")
wp_by_balls = df_all.groupby("balls_remaining")["win_probability"].mean()
print(wp_by_balls.head(6))
print("...")
print(wp_by_balls.tail(6))


print("\n✅ Full-match WPA computation complete!")


WPA VALIDATION

WPA sum per match:
  Mean: 0.0573 (should be ≈ 0.0)
  Std:  0.4422
  Min:  -1.8091
  Max:  1.4203

Balls by innings:
innings
1    706934
2    649249
dtype: int64

Total WPA by innings:
innings
1     47.080773
2    287.433422
Name: wpa, dtype: float64

Terminal WP (last ball of match):
  Mean: 0.527 (should be ≈ 0.52)
  Std:  0.487

WPA distribution (should have heavy tails):
count    1.356183e+06
mean     2.466586e-04
std      3.022908e-02
min     -7.993356e-01
1%      -9.909809e-02
5%      -3.789657e-02
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
95%      3.480677e-02
99%      8.470383e-02
max      9.604751e-01
Name: wpa, dtype: float64

Win probability by balls remaining (first/last):
balls_remaining
1    0.380696
2    0.392637
3    0.401655
4    0.413936
5    0.424857
6    0.432879
Name: win_probability, dtype: float64
...
balls_remaining
114    0.489123
115    0.487045
116    0.484146
117    0.482022
118    0.479148
119    0.469703
Name: win_p

In [20]:
print("\nSaving ball-level WPA...")

output_cols = [
    "match_id", "innings", "over", "ball",
    "batting_team", "bowling_team",
    "batter", "bowler",
    "runs_off_bat", "extras", "total_runs", "is_wicket",
    "win_probability", "wpa",
    "date", "league", "venue"
]

df_all[output_cols].to_csv(
    "C:/Desktop/crickanal/outputs/ball_by_ball_wp_wpa_t20.csv",
    index=False
)

print(f"✅ Saved to: outputs/ball_by_ball_wp_wpa_t20.csv")


Saving ball-level WPA...
✅ Saved to: outputs/ball_by_ball_wp_wpa_t20.csv


In [21]:
print("\n" + "="*60)
print("PLAYER AGGREGATION - BATTING")
print("="*60)

# Aggregate batting WPA per player-match-innings
batting_agg = (
    df_all
    .groupby(["match_id", "innings", "batter", "date", "league", "venue", "batting_team"])
    .agg({
        "wpa": "sum",
        "runs_off_bat": "sum",
        "ball": "count",
        "is_wicket": lambda x: (x == 1).any()
    })
    .rename(columns={
        "wpa": "bat_wpa",
        "runs_off_bat": "runs_scored",
        "ball": "balls_faced",
        "is_wicket": "dismissed"
    })
    .reset_index()
    .rename(columns={"batter": "player"})
)

batting_agg["dismissed"] = batting_agg["dismissed"].astype(int)

print(f"\nBatting records: {len(batting_agg):,}")
print(f"Unique players: {batting_agg['player'].nunique()}")
print(f"\nTop 5 batting WPA performances:")
print(batting_agg.nlargest(5, "bat_wpa")[["player", "date", "runs_scored", "balls_faced", "bat_wpa"]])


PLAYER AGGREGATION - BATTING

Batting records: 91,192
Unique players: 3510

Top 5 batting WPA performances:
                player       date  runs_scored  balls_faced   bat_wpa
28314      Vivek Kumar 2019-02-28           18            6  1.365458
88851  Ashutosh Sharma 2025-03-24           66           31  1.318041
72018        H Klaasen 2023-07-25          110           44  1.273970
80067   Shashank Singh 2024-04-04           61           30  1.252265
73668     MC Henriques 2024-01-16           43           20  1.231666


In [22]:
print("\n" + "="*60)
print("PLAYER AGGREGATION - BOWLING")
print("="*60)

# Aggregate bowling WPA per player-match-innings
# Note: bowler gets NEGATIVE of batting team's WPA gain
bowling_agg = (
    df_all
    .groupby(["match_id", "innings", "bowler", "date", "league", "venue", "bowling_team"])
    .agg({
        "wpa": lambda x: -x.sum(),  # Negative because bowling perspective
        "total_runs": "sum",
        "ball": "count",
        "is_wicket": "sum"
    })
    .rename(columns={
        "wpa": "bowl_wpa",
        "total_runs": "runs_conceded",
        "ball": "balls_bowled",
        "is_wicket": "wickets_taken"
    })
    .reset_index()
    .rename(columns={"bowler": "player"})
)

print(f"\nBowling records: {len(bowling_agg):,}")
print(f"Unique players: {bowling_agg['player'].nunique()}")
print(f"\nTop 5 bowling WPA performances:")
print(bowling_agg.nlargest(5, "bowl_wpa")[["player", "date", "wickets_taken", "runs_conceded", "bowl_wpa"]])


PLAYER AGGREGATION - BOWLING

Bowling records: 69,952
Unique players: 2646

Top 5 bowling WPA performances:
            player       date  wickets_taken  runs_conceded  bowl_wpa
810       DW Steyn 2014-03-24              4             17  1.157146
19741  L Trevaskis 2018-08-07              4             16  1.093479
13391    JJ Bumrah 2017-01-29              2             21  1.020711
7623     S Milenko 2016-01-13              3             19  0.999595
49547  B Swanepoel 2022-10-20              0             25  0.981225


In [23]:
print("\n" + "="*60)
print("MERGING BAT + BOWL IMPACTS")
print("="*60)

# Full outer merge
player_innings = batting_agg.merge(
    bowling_agg,
    on=["match_id", "innings", "player", "date", "league", "venue"],
    how="outer",
    suffixes=("_bat", "_bowl")
)

# Fill NaN for players who only batted or only bowled
player_innings["bat_wpa"] = player_innings["bat_wpa"].fillna(0)
player_innings["bowl_wpa"] = player_innings["bowl_wpa"].fillna(0)
player_innings["runs_scored"] = player_innings["runs_scored"].fillna(0).astype(int)
player_innings["balls_faced"] = player_innings["balls_faced"].fillna(0).astype(int)
player_innings["dismissed"] = player_innings["dismissed"].fillna(0).astype(int)
player_innings["runs_conceded"] = player_innings["runs_conceded"].fillna(0).astype(int)
player_innings["balls_bowled"] = player_innings["balls_bowled"].fillna(0).astype(int)
player_innings["wickets_taken"] = player_innings["wickets_taken"].fillna(0).astype(int)

# Compute total WPA
player_innings["total_wpa"] = player_innings["bat_wpa"] + player_innings["bowl_wpa"]

# Consolidate team column
player_innings["team"] = player_innings["batting_team"].fillna(player_innings["bowling_team"])
player_innings = player_innings.drop(columns=["batting_team", "bowling_team"], errors="ignore")

print(f"\nTotal player-innings records: {len(player_innings):,}")
print(f"Records with batting only: {(player_innings['balls_bowled'] == 0).sum():,}")
print(f"Records with bowling only: {(player_innings['balls_faced'] == 0).sum():,}")
print(f"Records with both bat+bowl: {((player_innings['balls_faced'] > 0) & (player_innings['balls_bowled'] > 0)).sum():,}")

print(f"\nTop 5 total WPA performances (bat+bowl):")
print(player_innings.nlargest(5, "total_wpa")[["player", "date", "runs_scored", "wickets_taken", "bat_wpa", "bowl_wpa", "total_wpa"]])


MERGING BAT + BOWL IMPACTS

Total player-innings records: 161,144
Records with batting only: 91,192
Records with bowling only: 69,952
Records with both bat+bowl: 0

Top 5 total WPA performances (bat+bowl):
                 player       date  runs_scored  wickets_taken   bat_wpa  \
50488       Vivek Kumar 2019-02-28           18              0  1.365458   
157037  Ashutosh Sharma 2025-03-24           66              0  1.318041   
127675        H Klaasen 2023-07-25          110              0  1.273970   
141702   Shashank Singh 2024-04-04           61              0  1.252265   
130615     MC Henriques 2024-01-16           43              0  1.231666   

        bowl_wpa  total_wpa  
50488        0.0   1.365458  
157037       0.0   1.318041  
127675       0.0   1.273970  
141702       0.0   1.252265  
130615       0.0   1.231666  


In [24]:
print("\nSaving player aggregation...")

output_cols_player = [
    "match_id", "innings", "player", "team", "date", "league", "venue",
    "runs_scored", "balls_faced", "dismissed",
    "runs_conceded", "balls_bowled", "wickets_taken",
    "bat_wpa", "bowl_wpa", "total_wpa"
]

player_innings[output_cols_player].to_csv(
    "C:/Desktop/crickanal/outputs/player_daily_innings_impact.csv",
    index=False
)

print(f"✅ Saved to: outputs/player_daily_innings_impact.csv")

# Summary stats
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Ball-level records: {len(df_all):,}")
print(f"Player-innings records: {len(player_innings):,}")
print(f"Unique players: {player_innings['player'].nunique()}")
print(f"Unique matches: {player_innings['match_id'].nunique()}")
print(f"Date range: {player_innings['date'].min()} to {player_innings['date'].max()}")
print(f"\nTotal WPA stats:")
print(player_innings["total_wpa"].describe())

print("\n" + "="*60)
print("✅✅✅ PIPELINE COMPLETE")


Saving player aggregation...
✅ Saved to: outputs/player_daily_innings_impact.csv

FINAL SUMMARY
Ball-level records: 1,356,183
Player-innings records: 161,144
Unique players: 3825
Unique matches: 5842
Date range: 2014-01-01 00:00:00 to 2025-07-26 00:00:00

Total WPA stats:
count    1.611440e+05
mean    -9.473245e-21
std      1.221068e-01
min     -1.265894e+00
25%     -5.002113e-02
50%     -2.072152e-03
75%      3.486406e-02
max      1.365458e+00
Name: total_wpa, dtype: float64

✅✅✅ PIPELINE COMPLETE
